# Data Exploration

In [10]:
!pip install -q python-terrier ir_datasets pandas

In [ ]:
import pyterrier as pt
import pandas as pd

#collecting from trec-covid round5 dataset
dataset_name = "irds:cord19/trec-covid/round5"
dataset = pt.get_dataset(dataset_name)

print(f"Loaded dataset: {dataset_name}")
irds_dataset = dataset.irds_ref()

Loaded dataset: irds:cord19/trec-covid/round5


## Query Exploration

In [16]:
topics_data = []
for query in irds_dataset.queries_iter():
    topics_data.append({
        # 'qid': query.query_id,
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })

df_topics = pd.DataFrame(topics_data)
print(f"Total number of queries: {len(df_topics)}")
df_topics.head(3)

Total number of queries: 50


,title,description,narrative
0,coronavirus origin,what is the origin of COVID-19,seeking range of information about the SARS-Co...
1,coronavirus response to weather changes,how does the coronavirus respond to changes in...,seeking range of information about the SARS-Co...
2,coronavirus immunity,will SARS-CoV2 infected people develop immunit...,seeking studies of immunity developed due to i...


## Relevance Judgments Exploration

In [17]:
qrels = dataset.get_qrels()

print(f"Total number of relevance judgments: {len(qrels)}")

print("\nRelevance Score Distribution:")
qrels['label'].value_counts()

[INFO] [starting] https://ir.nist.gov/covidSubmit/data/qrels-covid_d5_j4.5-5.txt
[INFO] [finished] https://ir.nist.gov/covidSubmit/data/qrels-covid_d5_j4.5-5.txt: [00:00] [402kB] [24.1MB/s]
                                                                                          

Total number of relevance judgments: 23151

Relevance Score Distribution:


label
 0    12239
 2     6677
 1     4233
-1        2
Name: count, dtype: int64

## Document Exploration

In [36]:
doc_iter = irds_dataset.docs_iter()

sample_docs = []
for i, doc in enumerate(doc_iter):
    if i >= 5:
        break
    sample_docs.append({
        'docno': doc.doc_id,
        'title': doc.title,
        'abstract': doc.abstract[:300]
    })

df_docs = pd.DataFrame(sample_docs)
df_docs

,docno,title,abstract
0,ug7v899j,Clinical features of culture-proven Mycoplasma...,OBJECTIVE: This retrospective chart review des...
1,02tnwd4m,Nitric oxide: a pro-inflammatory mediator in l...,Inflammatory diseases of the respiratory tract...
2,ejv2xln0,Surfactant protein-D and pulmonary host defense,Surfactant protein-D (SP-D) participates in th...
3,2b73a28n,Role of endothelin-1 in lung disease,Endothelin-1 (ET-1) is a 21 amino acid peptide...
4,9785vg6d,Gene expression in epithelial cells in respons...,Respiratory syncytial virus (RSV) and pneumoni...


## Indexing

In [38]:
import os

if not os.path.exists("./index/data.properties"):
    indexer = pt.IterDictIndexer("./index")
    
    index_ref = indexer.index(
        ({
            "docno": d.doc_id,
            "text": (d.title or "") + " " + (d.abstract or "")
        }
        for d in irds_dataset.docs_iter())
    )
else:
    index_ref = "./index"

index = pt.IndexFactory.of(index_ref)

In [39]:
#loading the index
index = pt.IndexFactory.of("./cord19_index/data.properties")
#or
#index = pt.IndexFactory.of(index_ref)